# Lab 3: The Stress Test

In Lab 1 we trained a neural network on **tabular** data to predict diabetes. In Lab 2 we used language models to extract variables from **text**. Today we work with a third data type: **images**.

This morning we learned that accuracy alone does not tell us if a model is trustworthy. We discussed explainability methods and the dangers of bias. Now let's put that knowledge to the test.

Keep in mind that when you are running code in Google Colab, you are running code in the cloud. Do not process private medical data in Google Colab!

## The Scenario

We will be working with images of skin lesions. Melanoma is the most dangerous form of skin cancer, early detection can be life-saving. The promise of AI here is to help dermatologists screen patients faster and more consistently. Today we will build a computer vision model to help detect malicious moles. We will then test the validity of our model using feature attribution via a heatmap.

In [ ]:
%pip install -q grad-cam
import torch, numpy, matplotlib.pyplot as plt

We have abstracted some code in a helper file for this tutorial. Load it in now.

In [ ]:
import os
if not os.path.exists('/content/ML4Epidemiology'):
    !git clone https://github.com/SimonIlic/ML4Epidemiology.git
%cd /content/ML4Epidemiology/Lesson3

In [ ]:
import Lesson3.lesson3_helper as lesson3_helper
train_dataset, test_dataset = lesson3_helper.load_data()

### The Dataset
Our dataset has been specifically curated for today's lesson.

Our data comes from the [ISIC Skin Lesion Archive](https://www.isic-archive.com/), one of the largest public collections of dermatology images. Each image shows a skin lesion and has been labeled by a dermatologist as either **benign** (harmless) or **malignant** (cancerous). Our task is simple: build a model that classifies skin lesions into these two categories.

## Exploring the Dataset

Let's look at some example images from our dataset. Benign lesions are shown in green, malignant in red.

In [ ]:
lesson3_helper.show_images(train_dataset, n=8)

### Exercise 1
Browse through the images by running the cell above multiple times (it shows random samples each time). What visual differences do you notice between benign and malignant samples? Write your observations below.

...

Explore the dataset further. How many test and training samples do we have? What is the class distribution?

## The Model: Convolutional Neural Networks

In Lab 1 our neural network read numbers from a table. A **Convolutional Neural Network (CNN)** reads *images*. It scans the image with small filters to detect patterns — first simple edges, then shapes, then textures, then complex structures. Layer by layer it builds up an understanding of what it sees, similar to how human vision first notices color irregularities, then border shapes, then overall structure.

We are not going to build a CNN from scratch. Instead we will use an approach called **Transfer Learning**.

### Transfer Learning

Training a CNN from scratch requires millions of images and days of computation. Instead, we take a model that has already been trained on millions of everyday photos (cats, cars, buildings, a dataset called ImageNet) and retrain *just the final decision layer* on our medical images. 

Think of it like hiring an experienced pathologist who already knows how to analyze tissue samples, and then training them specifically on skin lesions. They don't need to relearn what a cell looks like, they just need to learn which patterns indicate cancer.

We will use **MobileNetV2**, a lightweight CNN developed by Google.

In [ ]:
from transformers import AutoImageProcessor, MobileNetV2ForImageClassification
import torch.nn as nn

# Load a pre-trained CNN
model_name = "google/mobilenet_v2_1.0_224"
processor = AutoImageProcessor.from_pretrained(model_name)
model = MobileNetV2ForImageClassification.from_pretrained(model_name)

# Freeze all the pre-trained layers, we keep the "experience" intact
for param in model.base_model.parameters():
    param.requires_grad = False

# Replace only the final classifier layer for our binary task
output_dim = ...  # How many outputs do we need for benign vs malignant?

model.classifier = nn.Sequential(
    nn.Linear(1280, output_dim),
    nn.Sigmoid()
)

Let's see how many parameters this model has, and how many we are actually training. Most of the model is frozen, we are only updating the final layer.

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")

## Training

Just like in Lab 1, we need a **loss function** (to measure how wrong the model is) and an **optimizer** (to nudge the weights in the right direction). Since this is a binary classification task we use the same Binary Cross Entropy loss as before.

The one new concept here is a **DataLoader**. When working with images we cannot feed the entire dataset to the model at once (it would run out of memory). Instead, we feed it small batches of 16 images at a time.

Based on what you learned in tutorial 1 choose a criterion and optimizer.

In [ ]:
import torch.optim as optim

# DataLoaders feed images to the model in batches
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False)

criterion = ...
optimizer = ...

In [ ]:
# Training loop
losses = []
model.train()

for epoch in range(5):
    epoch_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images).logits.squeeze(1)
        loss = criterion(outputs, labels.float())
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/5 - Loss: {avg_loss:.4f}")

In [ ]:
plt.plot(losses, marker='o')
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

## Evaluation: Standard Metrics

The model is trained! Let's run the standard evaluation. As you learned this morning, we look at accuracy, sensitivity and specificity.

In [ ]:
# Evaluate on the test set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images).logits.squeeze(1)
        predictions = (outputs > 0.5).int()
        all_preds.extend(predictions.tolist())
        all_labels.extend(labels.tolist())

accuracy = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"Test Accuracy: {accuracy:.2%}")

In [ ]:
import pandas
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)
cm_df = pandas.DataFrame(cm,
    index=["True Benign", "True Malignant"],
    columns=["Predicted Benign", "Predicted Malignant"])
cm_df

In [ ]:
# Show some test predictions with confidence
model.eval()
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = [ax for row in axes for ax in row]

indices = numpy.random.choice(len(test_dataset), 8, replace=False)
for i, idx in enumerate(indices):
    img_tensor, true_label = test_dataset[idx]
    with torch.no_grad():
        confidence = model(img_tensor.unsqueeze(0)).logits.item()
    pred_label = 1 if confidence > 0.5 else 0
    
    axes[i].imshow(lesson3_helper._denormalize(img_tensor))
    true_name = lesson3_helper.CLASS_NAMES[true_label]
    pred_name = lesson3_helper.CLASS_NAMES[pred_label]
    color = 'green' if pred_label == true_label else 'red'
    axes[i].set_title(f"True: {true_name}\nPred: {pred_name} ({confidence:.0%})", fontsize=9, color=color)
    axes[i].axis('off')

plt.suptitle("Model Predictions on Test Images", fontsize=14)
plt.tight_layout(); plt.show()

### Exercise 2

Based on these metrics, would you sign off on deploying this model in a hospital to help dermatologists screen patients? Write your recommendation below.

...

## The Audit: Looking Under the Hood

Hold on. As we discussed in the lecture, metrics tell us *how well* the model performs, but not *why*. Before signing off on deployment, we should look under the hood. We need to know: **what is the model actually looking at when it makes a prediction?**

For this we use an **explainability** technique called **Grad-CAM** (Gradient-weighted Class Activation Mapping).

### What is Grad-CAM?

Grad-CAM creates a **heatmap** overlay on the original image. It highlights which regions of the image drove the model's prediction:

- **Red/Yellow** = The model focused heavily on this area
- **Blue/Green** = The model mostly ignored this area

If the model has truly learned to detect cancer, we would expect the heatmap to light up over the skin lesion itself. Let's check.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# Set up Grad-CAM on the last convolutional layer of MobileNetV2
target_layers = [model.base_model.layer[-1]]  # Last conv layer in MobileNetV2
# We wrap the model to ensure Grad-CAM works with the HF model's output
wrapped_model = lesson3_helper.ModelWrapper(model)
cam = GradCAM(model=wrapped_model, target_layers=target_layers)

model.eval()

# Unfreeze all parameters so Grad-CAM can compute the backward pass
for param in model.base_model.parameters():
    param.requires_grad = True

In [ ]:
# Generate Grad-CAM heatmaps for images predicted as malignant
model.eval()

# Find images the model predicted as malignant with high confidence
malignant_indices = []
for idx in range(len(test_dataset)):
    img_tensor, label = test_dataset[idx]
    with torch.no_grad():
        conf = model(img_tensor.unsqueeze(0)).logits.item()
    if conf > 0.5:  # model thinks this is malignant
        malignant_indices.append((idx, conf))

# Pick the top 6 by confidence
malignant_indices = sorted(malignant_indices, key=lambda x: -x[1])[:6]
n_show = len(malignant_indices)
print(f"Showing Grad-CAM for {n_show} images predicted as malignant")

fig, axes = plt.subplots(n_show, 2, figsize=(8, 3 * n_show))
if n_show == 1:
    axes = [axes]

for i, (idx, conf) in enumerate(malignant_indices):
    img_tensor, true_label = test_dataset[idx]
    rgb_img = lesson3_helper._denormalize(img_tensor)
    
    # Generate heatmap
    input_tensor = img_tensor.unsqueeze(0)
    grayscale_cam = cam(input_tensor=input_tensor)
    heatmap = show_cam_on_image(rgb_img, grayscale_cam[0], use_rgb=True)
    
    # Plot original and heatmap side by side
    axes[i][0].imshow(rgb_img)
    axes[i][0].set_title(f"Original (conf: {conf:.0%})")
    axes[i][0].axis('off')
    
    axes[i][1].imshow(heatmap)
    axes[i][1].set_title("Grad-CAM Heatmap")
    axes[i][1].axis('off')

plt.suptitle("What is the model looking at?", fontsize=14)
plt.tight_layout(); plt.show()

### Look carefully at the heatmaps above.

**What is the model actually focusing on to make its cancer predictions?**

### Exercise 3

The model is not detecting cancer: it is detecting **rulers**!

Answer the following questions:

a) Why might rulers appear more often in images of suspicious (malignant) moles?

b) How does this connect to the concept of **confounders** from epidemiology?

c) Has your recommendation from Exercise 2 changed? Would you still deploy this model?

...

## What Went Wrong: The "Clever Hans" Effect

This is one of the most famous cautionary tales in AI. It is called the **"Clever Hans" effect**, [named after a horse in the 1900s](https://en.wikipedia.org/wiki/Clever_Hans) that appeared to solve math problems, but was actually reading its trainer's body language.

Here is what happened: In the real [ISIC skin lesion dataset](https://www.isic-archive.com/), when a dermatologist sees a suspicious mole, they place a millimeter ruler next to it to document its size. Benign moles usually do not get the ruler treatment. The AI discovered that predicting "cancer" whenever it sees a ruler gives it a high accuracy score **without ever learning what cancer looks like**.

For this tutorial we artificially added rulers to the data, but this effect has been documented in real published research:
- Bissoto et al. (2019) blacked out the actual skin lesions entirely. The model *still predicted cancer at high accuracy* just by looking at rulers and band-aids in the background.
- Winkler et al. (2019) drew purple surgical pen marks next to benign moles. The AI's predicted probability of melanoma falsely skyrocketed.

In machine learning this is called **shortcut learning**: the model found an easy shortcut (ruler detection) instead of learning the hard task (cancer detection). The model learned a **spurious association**, not the causal mechanism.

## Why This Matters

If a clinic deployed this model:
- A patient with **deadly melanoma** might be sent home with a clean bill of health simply because the doctor did not put a ruler in the photo.
- A patient with a **harmless mole** might be rushed to surgery because the photo happened to include a ruler.

This is not a one-off failure. Similar shortcut learning has been found across medical AI:
- **COVID-19 X-rays**: A model learned to identify COVID by reading the *font on the X-ray machine* (different hospitals used different machines, and some had more COVID patients).
- **Surgical markings**: Purple pen marks next to benign moles caused false positive melanoma predictions (Winkler et al., 2019).
- **Wolf vs. Husky**: A famous image classifier distinguished wolves from huskies purely by whether there was *snow in the background*.

### Exercise 4

How would you fix this model? List at least 3 approaches you would try.

...

### Exercise 5: Connecting to Epidemiology

Imagine this model was trained on data from **Hospital A** (which routinely photographs suspicious moles with rulers) and deployed at **Hospital B** (which does not use rulers in their photos).

a) What would happen to the model's performance at Hospital B?

b) How is this similar to the concept of **domain shift**?

...

## Conclusion

You have learned the most important lesson in medical AI: **accuracy is necessary but not sufficient**. A model can achieve impressive metrics while learning completely the wrong thing.

Always audit *what* your model has learned before trusting its predictions. Explainability tools like Grad-CAM are essential in high-stakes domains.

**Key takeaways:**
1. High accuracy does not mean a model is correct the way we may think. It may have learned a shortcut.
2. Explainability methods (like Grad-CAM) reveal what the model actually looks at.
3. Data bias can silently undermine even the best model architectures.
4. External validation across different settings is critical before deployment.

**To explore further:**
1. Try removing the images with rulers from the training set and retraining. Does the model learn something different?
2. Try adding rulers to the benign images too. What happens to accuracy?
3. Read the original papers: [Bissoto et al. 2019](https://arxiv.org/abs/1904.07475) and [Winkler et al. 2019](https://jamanetwork.com/journals/jamadermatology/fullarticle/2740808).